***

# **EDU Data (Edu_2, Edu_3)**

***

This file is dedicated to converting Seth's work on the Edu_2 and Edu_3 indicators from R to Python. The work Seth did is still up to date, all we want to do is make it uniform with the rest of the work we're doing.

***

## **Packages**

***

In [2]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import os

***

## **Processing**

***

In [3]:
# Datasets are from 2017-2023 and are contained in URLs

urls = [
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr23.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr22.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr21.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr20.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr19.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr18.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr17.txt"
]

# Read data from URLs into dfs
dataframes = [pd.read_csv(url, sep="\t", header=0) for url in urls]

C:\Users\jchoy\AppData\Local\Temp\ipykernel_18008\3552500611.py:14: DtypeWarning: Columns (34,35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(url, sep="\t", header=0) for url in urls]


In [4]:
# List of col names

nms = [
    "AcYear", "AggLevel", "CountyCode", "DistCode",
    "SchoolCode", "CountyName", "DistName", "SchoolName",
    "Charter", "DASS", "RepCat", "Cohort", "HSDiploma_Ct",
    "HSDiploma_Pct", "AG_Ct", "AG_Pct", "Biliteracy_Ct",
    "Biliteracy_Pct", "GSSMerit_Ct", "GSSMerit_Pct",
    "CHSPE_Ct", "CHSPE_Pct", "AdEdDiploma_Ct",
    "AdEdDiploma_Pct", "SPED_Ct", "SPED_Pct",
    "GED_Ct", "GED_Pct", "OtherTransfer_Ct",
    "OtherTransfer_Pct", "Dropout_Ct", "Dropout_Pct",
    "StillEnrolled_Ct", "StillEnrolled_Pct"
]


# 2021-2022 does not have the same cols as others, so we drop the ones that don't match

columns_df1 = set(dataframes[2].columns)
columns_df2 = set(dataframes[1].columns)

# Identify cols in df2 not in df1

extra_columns = columns_df2 - columns_df1

# Drop extra cols from dataframes

dataframes[1] = dataframes[1].drop(columns=extra_columns)

# Renaming cols names of dataframes

for df in dataframes:
    df.columns = nms

# Reversing list

dataframes.reverse()

# By reversing, we can initialize using 2016-2017

schools = dataframes[0]

# Append the remaining dfs

for df in dataframes[1:6]:
    schools = pd.concat([schools, df], ignore_index=True)

# Replace empty strings and '*' with NaN
schools.replace({"": np.nan, "*": np.nan}, inplace=True)

# List of SACOG counties

sacog_counties = [
    "El Dorado", "Placer", "Sacramento",
    "Sutter", "Yolo", "Yuba"
]

# List of categories

cats = ["RB", "RA", "RF", "RH", "RD", "RP", "RT", "RW", "TA", "SS"]

In [5]:
# Select categorical columns

schools_cat = schools.loc[:, "AcYear":"RepCat"]

# Convert specified columns to num

numeric_columns = ["Cohort", "HSDiploma_Ct", "HSDiploma_Pct", "AG_Ct", "AG_Pct", "Biliteracy_Ct",
                   "Biliteracy_Pct", "GSSMerit_Ct", "GSSMerit_Pct", "CHSPE_Ct", "CHSPE_Pct",
                   "AdEdDiploma_Ct", "AdEdDiploma_Pct", "SPED_Ct", "SPED_Pct", "GED_Ct", "GED_Pct",
                   "OtherTransfer_Ct", "OtherTransfer_Pct", "Dropout_Ct", "Dropout_Pct", "StillEnrolled_Ct",
                   "StillEnrolled_Pct"]

schools_num = schools.loc[:, numeric_columns].apply(pd.to_numeric, errors='coerce')

# Combine the categorical and num

schools_combined = pd.concat([schools_cat, schools_num], axis=1)

# Filter the combined df

sacog_schools = schools_combined[
    (schools_combined["CountyName"].isin(sacog_counties)) &
    (schools_combined["AggLevel"] == "D") &
    (schools_combined["Charter"] == "All") &
    (schools_combined["DASS"] == "All") &
    (schools_combined["RepCat"].isin(cats)) &
    (~schools_combined["HSDiploma_Pct"].isna())
]

# Display the filtered DataFrame

display(sacog_schools.head(5))

,AcYear,AggLevel,CountyCode,DistCode,SchoolCode,CountyName,DistName,SchoolName,Charter,DASS,...,SPED_Ct,SPED_Pct,GED_Ct,GED_Pct,OtherTransfer_Ct,OtherTransfer_Pct,Dropout_Ct,Dropout_Pct,StillEnrolled_Ct,StillEnrolled_Pct
12361,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,13.0,9.0,39.1
12363,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,1.0,1.4,2.0,2.8,9.0,12.7,16.0,22.5
12367,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,4.0,3.8,4.0,3.8,24.0,23.1,15.0,14.4
12373,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,4.0,2.4,5.0,3.0,38.0,23.2,37.0,22.6
12374,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,5.0,2.3,6.0,2.8,42.0,19.5,43.0,20.0


In [6]:
# Select and rename RepCat values for sacog_ag

sacog_ag = sacog_schools.loc[:, ["AcYear", "CountyName", "DistName", "RepCat", "AG_Ct", "AG_Pct"]]
sacog_ag['RepCat'] = sacog_ag['RepCat'].replace({
    "RB": "Black",
    "RI": "Indigenous",
    "RA": "Asian",
    "RF": "Filipino",
    "RH": "Latino",
    "RD": "Not Reported",
    "RP": "Pacific Islander",
    "RT": "Two or More",
    "RW": "White",
    "TA": "Total",
    "SS": "Socioeconomically Disadvantaged"
})

# Filter and select specific columns for sacog_county

sacog_county = schools[
    (schools["CountyName"].isin(sacog_counties)) &
    (schools["AggLevel"] == "C") &
    (schools["Charter"] == "All") &
    (schools["DASS"] == "All") &
    (schools["RepCat"].isin(cats)) &
    (~schools["HSDiploma_Pct"].isna())
]

sacog_ag_cty = sacog_county.loc[:, ["AcYear", "CountyName", "DistName", "RepCat", "AG_Ct", "AG_Pct"]]
sacog_ag_cty['RepCat'] = sacog_ag_cty['RepCat'].replace({
    "RB": "Black",
    "RI": "Indigenous",
    "RA": "Asian",
    "RF": "Filipino",
    "RH": "Latino",
    "RD": "Not Reported",
    "RP": "Pacific Islander",
    "RT": "Two or More",
    "RW": "White",
    "TA": "Total",
    "SS": "Socioeconomically Disadvantaged"
})

sacog_ag_cty['DistName'] = "County Total"

# Combine the dfs

sacog_combined = pd.concat([sacog_ag_cty, sacog_ag])

# Arrange by AcYear and CountyName

sacog_combined = sacog_combined.sort_values(by=["AcYear", "CountyName"])

In [7]:
sacog_combined.head(10)

,AcYear,CountyName,DistName,RepCat,AG_Ct,AG_Pct
890,2016-17,El Dorado,County Total,Asian,69,83.1
891,2016-17,El Dorado,County Total,Black,9,26.5
893,2016-17,El Dorado,County Total,Filipino,13,39.4
894,2016-17,El Dorado,County Total,Latino,96,29.2
897,2016-17,El Dorado,County Total,Two or More,47,54.0
898,2016-17,El Dorado,County Total,White,763,52.3
904,2016-17,El Dorado,County Total,Socioeconomically Disadvantaged,156,23.9
905,2016-17,El Dorado,County Total,Total,1010,49.2
12361,2016-17,El Dorado,El Dorado County Office of Education,Black,0.0,0.0
12363,2016-17,El Dorado,El Dorado County Office of Education,Latino,0.0,0.0


***

## **BEA Output_1**

***

In [3]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_data = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators','Seth', 'BEA', 'BEAbyIndustry.csv')
path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_edu  = os.path.join(path_git, 'Python Code', 'EDU')

<>:5: SyntaxWarning: invalid escape sequence '\R'
<>:5: SyntaxWarning: invalid escape sequence '\R'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_10132\3222863983.py:5: SyntaxWarning: invalid escape sequence '\R'
  path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')


In [4]:
# def notin(element, collection):
#     return element not in collection

sacog = ["Yuba City", "Sacramento-Roseville-Folsom"]
mtc = ["San Francisco-Oakland-Berkeley", "Santa Rosa-Petaluma",
       "Vallejo", "Napa", "San Jose-Sunnyvale-Santa Clara"]
scag = ["Los Angeles-Long Beach-Anaheim",
        "Riverside-San Bernardino-Ontario",
        "Oxnard-Thousand Oaks-Ventura",
        "El Centro"]

rdp = pd.read_csv(path_data, skiprows=3, na_values="(D)")


In [147]:
rdp.columns = rdp.columns.str.strip()

rdp.columns

Index(['GeoFips', 'GeoName', 'LineCode', 'Level', 'Description', '2017',
       '2018', '2019', '2020', '2021', '2022'],
      dtype='object')

In [148]:
# Filtering and selecting columns

rdp = rdp[rdp['LineCode'] < 87]

# rdp[['GeoName', 'Level']]
# rdp = rdp[rdp['GeoName']]
#rdp = rdp.loc[:, ['GeoName', 'Level'] + [str(year) for year in range(2018, 2023)]]
rdp = rdp.dropna(subset=['GeoName'])
rdp = rdp[rdp['Description'] != 'Addenda:']

# Replacing parts of GeoName

rdp['GeoName'] = rdp['GeoName'].str.replace(r"\(.*", "", regex=True)
rdp['GeoName'] = rdp['GeoName'].str.replace(r",.*", "", regex=True)

rdp['MPO'] = np.where(rdp['GeoName'].isin(sacog), 'SACOG', np.where(rdp['GeoName'].isin(mtc), "MTC", np.where(rdp['GeoName'].isin(scag), 'SCAG', np.where(rdp['GeoName'] == "San Diego-Chula Vista-Carlsbad", "SANDAG", rdp['GeoName'].str.replace('-',' ').str.split().str[0]))))
rdp['MPO'] = rdp['MPO'].str.replace(r"\(.*", "", regex=True)
rdp['MPO'] = rdp['MPO'].str.replace(r",.*", "", regex=True)

In [149]:
rdp = rdp[["GeoName", "MPO", "Level", "Description", "2017", "2018", "2019", "2020", "2021", "2022"]]

In [150]:
rdp.columns = ["MSA", "MPO", "Level", "Industry", "2017", "2018", "2019", "2020", "2021", "2022"]

In [151]:
rdp

,MSA,MPO,Level,Industry,2017,2018,2019,2020,2021,2022
0,Austin-Round Rock-Georgetown,Austin,1.0,All industry total,141102903.0,151747891.0,164432513.0,171619050.0,195828575.0,222054436.0
1,Austin-Round Rock-Georgetown,Austin,2.0,Private industries,124745562.0,134683231.0,146802709.0,152786094.0,176402729.0,201700555.0
2,Austin-Round Rock-Georgetown,Austin,3.0,"Agriculture, forestry, fishing and hunting",31422.0,45787.0,30722.0,37701.0,37372.0,39393.0
3,Austin-Round Rock-Georgetown,Austin,3.0,"Mining, quarrying, and oil and gas extraction",978382.0,1114801.0,1296493.0,1328372.0,984445.0,1045704.0
4,Austin-Round Rock-Georgetown,Austin,3.0,Utilities,687391.0,NaN,NaN,NaN,983376.0,1178042.0
...,...,...,...,...,...,...,...,...,...,...
933,Yuba City,SACOG,3.0,"Arts, entertainment, recreation, accommoda...",167914.0,179715.0,194110.0,166029.0,224392.0,263971.0
934,Yuba City,SACOG,4.0,"Arts, entertainment, and recreation",22904.0,24764.0,23434.0,16003.0,22735.0,30626.0
935,Yuba City,SACOG,4.0,Accommodation and food services,145010.0,154951.0,170676.0,150026.0,201657.0,233345.0
936,Yuba City,SACOG,3.0,Other services (except government and gove...,131340.0,139235.0,144718.0,137039.0,NaN,154873.0


In [152]:
rdp_long = pd.melt(rdp.reset_index(), id_vars=['MSA', 'MPO', 'Level', 'Industry'], value_vars=['2017', '2018', '2019', '2020', '2021', '2022'])

rdp_long.columns = ["MSA", "MPO", "Level", "Industry", "Year", "GRP"]

rdp_long = rdp_long[["Year", "MSA", "MPO", "Level", "Industry", "GRP"]]

In [153]:
rdp_long

,Year,MSA,MPO,Level,Industry,GRP
0,2017,Austin-Round Rock-Georgetown,Austin,1.0,All industry total,141102903.0
1,2017,Austin-Round Rock-Georgetown,Austin,2.0,Private industries,124745562.0
2,2017,Austin-Round Rock-Georgetown,Austin,3.0,"Agriculture, forestry, fishing and hunting",31422.0
3,2017,Austin-Round Rock-Georgetown,Austin,3.0,"Mining, quarrying, and oil and gas extraction",978382.0
4,2017,Austin-Round Rock-Georgetown,Austin,3.0,Utilities,687391.0
...,...,...,...,...,...,...
4531,2022,Yuba City,SACOG,3.0,"Arts, entertainment, recreation, accommoda...",263971.0
4532,2022,Yuba City,SACOG,4.0,"Arts, entertainment, and recreation",30626.0
4533,2022,Yuba City,SACOG,4.0,Accommodation and food services,233345.0
4534,2022,Yuba City,SACOG,3.0,Other services (except government and gove...,154873.0


In [154]:
rdp_long[~rdp_long["MPO"].isin(["MTC", "SANDAG", "SCAG", "SACOG"])]

,Year,MSA,MPO,Level,Industry,GRP
0,2017,Austin-Round Rock-Georgetown,Austin,1.0,All industry total,141102903.0
1,2017,Austin-Round Rock-Georgetown,Austin,2.0,Private industries,124745562.0
2,2017,Austin-Round Rock-Georgetown,Austin,3.0,"Agriculture, forestry, fishing and hunting",31422.0
3,2017,Austin-Round Rock-Georgetown,Austin,3.0,"Mining, quarrying, and oil and gas extraction",978382.0
4,2017,Austin-Round Rock-Georgetown,Austin,3.0,Utilities,687391.0
...,...,...,...,...,...,...
4475,2022,Tampa-St. Petersburg-Clearwater,Tampa,3.0,"Arts, entertainment, recreation, accommoda...",10117691.0
4476,2022,Tampa-St. Petersburg-Clearwater,Tampa,4.0,"Arts, entertainment, and recreation",2769188.0
4477,2022,Tampa-St. Petersburg-Clearwater,Tampa,4.0,Accommodation and food services,7348503.0
4478,2022,Tampa-St. Petersburg-Clearwater,Tampa,3.0,Other services (except government and gove...,4634669.0


In [155]:
sacog_grp = rdp_long[rdp_long['MPO'] == 'SACOG']

ca_peers = rdp_long[(rdp_long['MPO'] == 'MTC') | (rdp_long['MPO'] == 'SANDAG') | (rdp_long['MPO'] == 'SCAG')]

other_peers = rdp_long[~rdp_long["MPO"].isin(["MTC", "SANDAG", "SCAG", "SACOG"])]

dfs = [sacog_grp, ca_peers, other_peers, rdp_long]

In [156]:
sheet_names = ['SACOG GRP', 'CA MPO GRP', 'Peer MPO GRP', 'MSA GRP']

dataframe_path = os.path.join(path_config, 'BEA GRP Python.xlsx')

# Using ExcelWriter to write DataFrames to different sheets
with pd.ExcelWriter(dataframe_path, engine='openpyxl') as writer:
    for df, sheet_name in zip(dfs, sheet_names):
        df.to_excel(writer, index=False, sheet_name=sheet_name)